## F 검정
사례) 한 연구자는 두 가지 교육 방법 (A, B)이 학생들의 성적 분산에 차이가 있는지 알아보고자 한다. 각 집단에서 무작위로 10명씩 표본을 추출하여 시험 점수를 기록하였다.

In [2]:
import pandas as pd

df = pd.DataFrame({
    "A": [85, 88, 90, 78, 85, 86, 91, 89, 84, 87,
          88, 82, 85, 90, 86, 84, 87, 83, 89, 85],
    "B": [80, 76, 92, 85, 79, 94, 88, 90, 95, 77,
          91, 85, 96, 82, 97, 75, 93, 84, 90, 78]
})

(1) 통계절 가설 설정
- 귀무가설: 두 집단의 모분산은 같다.
- 대립가설: 두 집단의 모분산은 다르다.

(2) 검정 통계량 
- F 검정 통계량 공식: F = S^2(큰분산) / S^2(작은분산) -> S^2은 표본분산


In [4]:
var_A = df['A'].var()
var_B = df['B'].var()

if var_A > var_B:
    f_stat = var_A / var_B
else:
    f_stat = var_B / var_A

print("F-statistic:", f_stat)

F-statistic: 5.288213132400431


(3) p-value 구하기
- cdf(누적분포함수): 확률 변수가 특정 값보다 작거나 같을 확률을 나타내는 함수
1. F-통계량 계산
2. 자유도 계산
    - df1 = n큰분산 - 1
    - df2 = n작은분산 -1
3. p-value 구하기
    - 단측 검정: (좌측) p = CDF(F, df1, df2) , (우측) p = 1 - CDF(F, df1, df2)
    - 양측 검정: p = 2 x min(CDF(F, df1, df2), 1 - CDF(F, df1, df2))

In [5]:
from scipy import stats

if var_A > var_B:
    df1 = len(df['A']) - 1
    df2 = len(df['B']) - 1
else:
    df1 = len(df['B']) - 1
    df2 = len(df['A']) - 1

p_value = 2 * min(stats.f.cdf(f_stat, df1, df2), 1 - stats.f.cdf(f_stat, df1, df2))
print(p_value)

0.0006617791910468185


(해석) 귀무 가설을 기각 -> 두 집단 분산이 유의하게 다르다

### 일원분산분석(One-way ANOVA)
사례) 한 카페 체인에서는 3가지 원두 종류(브라질, 에티오피아, 콜롬비아)를 사용했을 때, 커피의 카페인 함량이 달라지는지 알아보려 한다. 각 원두로 커피를 8잔씩 추출해 카페인 함량을 측정하였다.

In [6]:
import pandas as pd

df = pd.DataFrame({
    "Brazil": [95, 97, 94, 100, 96, 98, 99, 97],
    "Ethiopia": [88, 90, 92, 89, 91, 90, 93, 89],
    "Colombia": [99, 101, 98, 100, 102, 97, 99, 101]
})

(1) 통계절 가설 설정
- 귀무가설: 세 원두의 평균 카페인 함량은 같다
- 대립가설: 적어도 한 원두의 평균 카페인 함량이 다르다.
- 독립성(기본 가정), 정규성(Shapiro-wilk), 등분산성(Levene) 가정

(2) 유의 수준 결정
- 유의수준 = 0.05

In [10]:
from scipy import stats

# 모두 모수 분포 (정규분포) 를 따른다
print(stats.shapiro(df["Brazil"]))
print(stats.shapiro(df["Ethiopia"]))
print(stats.shapiro(df["Colombia"]))

# 모두 등분산 이다
print(stats.levene(df["Brazil"], df["Ethiopia"], df["Colombia"]))

ShapiroResult(statistic=np.float64(0.9827990116364257), pvalue=np.float64(0.9754087657979972))
ShapiroResult(statistic=np.float64(0.9590108878016856), pvalue=np.float64(0.800626352774327))
ShapiroResult(statistic=np.float64(0.965656845766852), pvalue=np.float64(0.8619204744859971))
LeveneResult(statistic=np.float64(0.11731843575418995), pvalue=np.float64(0.8898807028666221))


(3) 검정방법
- 분산분석(ANOVA)는 집단 간 평균 차이를 F-통계량으로 평가한다.

In [11]:
stats.f_oneway(df["Brazil"], df["Ethiopia"], df["Colombia"])

F_onewayResult(statistic=np.float64(58.32467532467532), pvalue=np.float64(2.66792638066789e-09))

(해석) 귀무가설 기각 -> 세 원두의 평균 카페인 함량이 모두 같다고 보기 어렵다.